## GPT2 (Decoder)

GPT2 is a auto-regressive decoder-only transformer with was released by OpenAI in 2019. Models of this nature work from left to right to predict the next token based on all tokens that came before it. When these models are trained, a good chunk of internet is scrapped and then feed into some variant of the decoder-only block of the transformer. When pre-training is done, these models predict the next token very well, but the responses are not very clear.

Example of GPT2 with the Base model using examples from Gucci Mane's "Lemonade" lyrics.

### Example of GPT2 output with a higher temperature


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({
    "pad_token": "<|pad|>",
    "bos_token": "<|startoftext|>",
})

tokenizer.padding_size = "left"

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

phrases = [
    "My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon.",
    "I like them Georgia peaches but you look more like a lemon.",
    "I'm pimping wearing linen, that's just how I am chilling. I'm smoking grits and selling chickens. Corvette painted lemon.",
    "I got lemonade and lemon tint.",
    "Half a pound of lemon kush, call that pack the lemon drop.",
    "Just stash one lemon, homie, I can supply damn near 20 blocks.",
]

# tokenize phrases
inputs = tokenizer(phrases, return_tensors="pt", padding=True)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    num_return_sequences=15,
    pad_token_id=tokenizer.pad_token_id,
    temperature=0.7,
)


num_seqs = 15
for p_idx, phrase in enumerate(phrases):
    print(f"==== Phrase {p_idx} ====")
    for s_idx in range(num_seqs):
        out = outputs[p_idx * num_seqs + s_idx]
        print(f"--- sample {s_idx} ---")
        print(tokenizer.decode(out, skip_special_tokens=True))
    print()

/home/nick/github-projects/bert-t5-gpt2/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10413.64it/s]
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


==== Phrase 0 ====
--- sample 0 ---
My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon. that is a red. I am a real red. I am real red. You know, I am not. I am a real red. I am not in a red. I am not in a red. I am not in a red.
--- sample 1 ---
My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon. in a room full of people. A lemon is a very good idea, especially to do for a lot of people. I thought it was a lemon and it's a lemon. no no no no no no, it's a lemon.


--- sample 2 ---
My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon., I don't care what your fanny or your fanny says. What is the difference between fanny and the fanny? The difference between fanny and a fanny is what you are. You are a fanny because you want to
--- sample 3 ---
My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon. at the airport I went to this place with my wife 

### What the output means

Notice that with `temperature=0.7`, all 15 completions for each phrase are quite different from each other. For example, Phrase 0 generates varied continuations: some talk about clothes and denim, others about money, others apologize repeatedly. The model is sampling randomly from the probability distribution, so it explores different possible next tokens. The output doesn't make logical sense because GPT2 is just predicting the next likely word based on patterns in internet text—it has no understanding of the context or coherence.

Now let's see what happens when we lower the temperature to 0.1.

In [2]:
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    num_return_sequences=15,
    pad_token_id=tokenizer.pad_token_id,
    temperature=0.1,
)


num_seqs = 15
for p_idx, phrase in enumerate(phrases):
    print(f"==== Phrase {p_idx} ====")
    for s_idx in range(num_seqs):
        out = outputs[p_idx * num_seqs + s_idx]
        print(f"--- sample {s_idx} ---")
        print(tokenizer.decode(out, skip_special_tokens=True))
    print()

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


==== Phrase 0 ====
--- sample 0 ---
My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon., I am a little bit of a lecher than you.

I am a little bit of a lecher than you.
I am a little bit of a lecher than you.
I am a little bit of a lecher
--- sample 1 ---
My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon., I'm a little bit of a lecher than you.

I'm a little bit of a lecher than you.
I'm a little bit of a lecher than you.
I'm a little bit of a lecher
--- sample 2 ---
My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon.,

I'm a little bit of a leprech, but I'm not a lech.

I'm a little bit of a lech, I'm a little bit of a lech, I'm a little bit of
--- sample 3 ---
My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon.,

I'm a little bit of a sucker.

I'm a little bit of a sucker.

I'm a little bit of a sucker.

I'm a little bit of a sucker.

I

#### Explanation of the output

With `temperature=0.1`, notice that all 15 completions for each phrase are nearly identical. The model is being highly deterministic—it consistently selects the same most likely next token at each step, so it generates the same continuation over and over. This is the opposite of `temperature=0.7`, where the model explored many different possible continuations. 

The key takeaway: **temperature controls randomness**. Lower temperature makes the model more predictable and repetitive. Higher temperature makes it more diverse and exploratory. For a pre-trained model like GPT2 with no instruction-tuning, neither temperature produces coherent, meaningful text—it's just completing patterns from the training data.

**Note**: This is what output for a LLM looks like with no supervised fine-tuning.